In [ ]:
%%configure
{ "vCores": {  "parameterName": "pipelinecore",  "defaultValue": 8 }}

# What the arms are actually sitting on

`duckrun.get_stats(detailed=True)` reads the Delta log for the live file list and then every parquet
**footer**, returning one row per **column chunk per row group** -- `encodings`,
`dictionary_page_offset`, `total_compressed_size` and the min/max stats. That per-chunk detail is
the point: the aggregated form (`detailed=False`) reports `avg_row_group` and a compression string
and cannot answer either question this notebook exists for. It is also the only footer read here --
file counts, rows per file and row groups per file are all derived from those chunks.

**How much of the data is dictionary-encoded?** One number, by BYTES, off the `encodings` in the
footer: the chunk lists a dictionary encoding, or it does not. Nothing more is read into the list --
a bare `PLAIN` beside a dictionary means different things per writer (delta_rs puts one on every
dictionary chunk, because the dictionary page is itself PLAIN-encoded), so reading it as a fallback
signal scored that writer at zero. Whatever is not, VertiPaq has to build a
dictionary for at transcode -- time and memory on every cold read -- and that is the whole reason
the number is here. WHY a given column missed out is the writer's business and is not reported:
reading a reason out of an encoding list is guesswork.

**Did the ordering reach the files?** Row-group min/max on the first key: disjoint, monotonic ranges
mean whole row groups can be eliminated, overlapping ones mean they cannot, whatever the writer was
asked to do. The unsorted arm is expected to overlap on every neighbour -- it is the control.

**Everything computed here is written to the `tpcds_bench` lakehouse, Files section,
`Files/layout_stats/sf{sf}`** (raw chunks per arm as parquet, the classified chunks, and the summary
tables as CSV), so the numbers can be read outside Fabric: `python fabric/pull_layout_stats.py`.

**This is expensive** -- it opens every file of every arm -- and the layout only changes when the
tables are rebuilt. Run it once after a build, not on every results pass.

In [ ]:
!pip install -q duckrun --upgrade
notebookutils.session.restartPython()

In [ ]:
sf = 100
# Which arms to read. Drop names to make a run cheaper: every arm costs a full footer sweep of both
# facts. `default` is the control for the ordering question and is worth keeping in.
arms = "default,partition,cluster,vorder,vonly,duckdb"

In [ ]:
import os
import tempfile

import duckdb
import duckrun
import notebookutils
import pandas as pd

MIRROR_ITEM = "01b539f3-4a9d-45ef-b1ef-0ba59552eb21"   # mirrored Azure Databricks catalog
VORDER_LH = "tpcds_vorder"
BENCH_LH = "tpcds_bench"                                # results land in its Files section
# Scale factor is a PARAMETER, not a constant. Override `sf` in the parameters cell (or from a
# pipeline) and the schemas follow.
# `sf` and `arms` come from the parameters cell above.
RG_LO, RG_HI = 1_000_000, 16_000_000                   # Direct Lake's usable row-group window
FACTS = ("store_sales", "catalog_sales")
# First key of each arm's ordering -- the column whose row-group ranges say whether the ordering
# reached the files. `default` is expected to overlap on every neighbour; it is the control.
KEY = {"store_sales": "ss_sold_date_sk", "catalog_sales": "cs_sold_date_sk"}

ws_id = notebookutils.runtime.context["currentWorkspaceId"]
# GUIDs, never friendly names: this tenant has OneLake friendly-name support disabled.
vorder_id = notebookutils.lakehouse.get(VORDER_LH)["id"]
bench_id = notebookutils.lakehouse.get(BENCH_LH)["id"]
# arm -> (item holding it, schema). The mirrored Databricks arms all live in one item; the
# Fabric-written arms live in the tpcds_vorder lakehouse.
#
# The `duck*` arms are in tpcds_vorder too, and that is a name the lakehouse does not describe:
# they are written by delta_rs (delta-rs, via duckrun), NOT by the V-Order writer. The arm is the
# schema suffix, as everywhere here; the item is just where it is parked.
ALL_SESSIONS = {"default":    (MIRROR_ITEM, f"tpcds_sf{sf}_default"),
                "defaultf8": (MIRROR_ITEM, f"tpcds_sf{sf}_defaultf8"),
                "default2rg": (MIRROR_ITEM, f"tpcds_sf{sf}_default2rg"),
                "partition": (MIRROR_ITEM, f"tpcds_sf{sf}_partition"),
                "cluster":   (MIRROR_ITEM, f"tpcds_sf{sf}_cluster"),
                "clustersn": (MIRROR_ITEM, f"tpcds_sf{sf}_clustersn"),
                "vorder":    (vorder_id,   f"tpcds_sf{sf}"),
                "vonly":     (vorder_id,   f"tpcds_sf{sf}_vonly"),
                "duckdb":    (vorder_id,   f"tpcds_sf{sf}_duckdb"),
                "ducksort":  (vorder_id,   f"tpcds_sf{sf}_ducksort")}
_want = [a.strip() for a in arms.split(",") if a.strip()]
_bad = [a for a in _want if a not in ALL_SESSIONS]
if _bad:
    raise ValueError(f"unknown arm(s) {_bad}; expected any of {list(ALL_SESSIONS)}")
SESSIONS = tuple((a, *ALL_SESSIONS[a]) for a in _want)
print(f"sf{sf}, reading: " + ", ".join(f"{a} ({sch})" for a, _, sch in SESSIONS))

# Everything is staged on local disk first, then uploaded in one call at the end. The raw footer
# read goes to parquet straight from the DuckDB relation (exact types, no pandas round trip) --
# and `get_stats` is the ONE footer read per arm: the notebook then works off the local file, and
# everything below (file counts, rows per file, row groups per file) is derived from those chunks
# rather than re-opening the footers. No `parquet_file_metadata` pass: it re-opens every footer a
# second time over OneLake, one file at a time, and the only thing it adds is `created_by`, which
# says nothing the arm's name does not.
OUT_DIR = tempfile.mkdtemp(prefix=f"layout_stats_sf{sf}_")
OUT = f"layout_stats/sf{sf}"                            # relative to the lakehouse Files section

# Not every arm exists at every sf, so an arm whose schema is absent is reported and skipped rather
# than killing the notebook.
got = []
for arm, item, schema in SESSIONS:
    path = os.path.join(OUT_DIR, f"chunks_{arm}.parquet")
    try:
        sess = duckrun.connect(f"{ws_id}/{item}", schema=schema, name=arm)
        sess.get_stats(detailed=True).project(f"'{arm}' AS arm, *").write_parquet(path)
    except Exception as e:                                          # noqa: BLE001
        print(f"{arm:<8} no stats for schema {schema} ({str(e)[:110]})")
        continue
    got.append(arm)
    n, nf = duckdb.sql(f"SELECT count(*), count(DISTINCT file_name) "
                       f"FROM read_parquet('{path}')").fetchone()
    print(f"{arm:<8} {n:,} column chunks over {nf:,} files")
assert got, "no arm produced stats at this sf"

duckdb.sql(f"""
    CREATE OR REPLACE TABLE chunks AS
    SELECT * FROM read_parquet('{OUT_DIR}/chunks_*.parquet', union_by_name = true)
""")
# Per-file shape, straight off the chunks. A row group contributes one row per column, so the
# distinct (file, row_group_id) pairs are taken first -- summing row_group_num_rows over the raw
# chunks would multiply every count by the column count.
files_df = duckdb.sql("""
    WITH rg AS (SELECT DISTINCT arm, "table", file_name, row_group_id, row_group_num_rows AS rows
                FROM chunks)
    SELECT arm, "table", file_name, sum(rows) AS num_rows, count(*) AS num_row_groups
    FROM rg GROUP BY 1, 2, 3 ORDER BY 1, 2, 3
""").df()

In [ ]:
# One row per column chunk, classified. `encodings` is a comma-separated list, so it is split and
# trimmed rather than matched with LIKE: a bare LIKE '%PLAIN%' also matches PLAIN_DICTIONARY and
# would report every dictionary-encoded chunk as a fallback.
duckdb.sql("""
    CREATE OR REPLACE TABLE enc AS
    WITH e AS (
        SELECT arm, "table" AS tbl, path_in_schema AS col, file_name, row_group_id,
               row_group_num_rows AS rg_rows, total_compressed_size AS bytes,
               list_transform(str_split(coalesce(encodings, ''), ','), x -> trim(x)) AS encs
        FROM chunks
    )
    SELECT *,
           -- Dictionary-encoded, or not. A dictionary encoding in the list is the whole test, and
           -- it has to be, because a bare PLAIN alongside it means different things per writer:
           -- delta_rs lists PLAIN on EVERY dictionary chunk (the dictionary page itself is
           -- PLAIN-encoded), so excluding those scored the whole delta_rs family at zero. Measured
           -- on this project's exports, no parquet-mr chunk has ever carried both, so this is the
           -- same number as the old rule everywhere it was not simply wrong.
           list_contains(encs, 'RLE_DICTIONARY') OR list_contains(encs, 'PLAIN_DICTIONARY') AS is_dict
    FROM e
""")

print("--- Dictionary, by BYTES. Whatever is not dictionary-encoded gets its dictionary rebuilt by")
print("    VertiPaq on every cold read.")
summary_dict = duckdb.sql("""
    SELECT arm, tbl AS "table",
           round(100.0 * sum(bytes) FILTER (WHERE is_dict) / sum(bytes), 1) AS pct_bytes_dict,
           count(*) FILTER (WHERE NOT is_dict)              AS chunks_not_dict,
           count(*)                                         AS chunks,
           count(DISTINCT col) FILTER (WHERE NOT is_dict)   AS cols_not_dict,
           count(DISTINCT col)                              AS cols
    FROM enc GROUP BY 1, 2 ORDER BY 1, 2
""").df()
display(summary_dict)

# Every column of every table, facts first, ranked by MB -- this is what gets written out. The
# display below cuts it to what costs: the facts' non-dictionary columns, biggest first. A wide
# column missing its dictionary is what costs; a narrow one is noise.
columns_df = duckdb.sql(f"""
    SELECT arm, tbl AS "table", col, is_dict,
           count(*) AS chunks, round(sum(bytes) / 1048576.0, 1) AS mb,
           any_value(list_aggregate(encs, 'string_agg', ', ')) AS encodings
    FROM enc
    GROUP BY 1, 2, 3, 4 ORDER BY (tbl IN {FACTS}) DESC, arm, mb DESC
""").df()
print("--- The fact columns that are NOT dictionary-encoded.")
display(columns_df[columns_df["table"].isin(FACTS) & ~columns_df["is_dict"]])

In [ ]:
# Row-group geometry, from the same footer read. One row group per file is the shape the recipe
# targets; `pct_in_window` is the paper's own Table 9.3.2.1 metric. `spread` (max/min over the
# groups that are not the remainder) is the UNIFORMITY criterion -- ragged groups are the failure
# mode, and the cluster arm is the one at risk of them: it hits its row target with
# maxRecordsPerFile, which leaves one short tail per task.
print(f"--- Row groups. Direct Lake's window is {RG_LO:,}..{RG_HI:,} rows.")
row_groups = duckdb.sql(f"""
    WITH rg AS (SELECT DISTINCT arm, tbl, file_name, row_group_id, rg_rows FROM enc),
         r AS (SELECT *, count(*) OVER (PARTITION BY arm, tbl) AS n,
                      row_number() OVER (PARTITION BY arm, tbl ORDER BY rg_rows) AS smallest
               FROM rg)
    SELECT arm, tbl AS "table", count(DISTINCT file_name) AS files, count(*) AS row_groups,
           round(count(*)::DOUBLE / count(DISTINCT file_name), 2) AS rg_per_file,
           min(rg_rows) AS rg_min, round(avg(rg_rows)) AS rg_avg, max(rg_rows) AS rg_max,
           -- the single smallest group is the remainder bin and is judged apart, as in rowgroup_probe
           round(max(rg_rows)::DOUBLE / nullif(min(rg_rows) FILTER (WHERE smallest > 1 OR n = 1), 0), 3)
               AS spread,
           count(*) FILTER (WHERE rg_rows < {RG_LO}) AS groups_under_1m,
           round(100.0 * count(*) FILTER (WHERE rg_rows BETWEEN {RG_LO} AND {RG_HI}) / count(*), 1)
               AS pct_in_window
    FROM r GROUP BY 1, 2 ORDER BY 1, 2
""").df()
display(row_groups)

# Did the ordering reach the files? Row groups sorted by their low bound on the first key: if any
# range starts before its predecessor ends, that pair overlaps and a filter landing in the overlap
# eliminates neither. `default` is the control and should overlap nearly everywhere; `cluster` is
# the arm this measures -- 0 overlaps means clustering on write placed the rows, a high count means
# it did not and the arm measures nothing.
#
# A key that is a HIVE PARTITION column is not in the parquet file at all (the V-Order arm is
# partitioned by the date key), so there are no chunks and no stats to read. That is reported as
# `partitioned` -- a stronger form of elimination, not a failure -- and never as "0 row groups,
# not eliminable", which is what it used to look like.
print("--- Ordering on the first key. overlaps = neighbouring row-group ranges that intersect.")
key_rows = []
for arm, tbl in duckdb.sql(
        f"SELECT DISTINCT arm, tbl FROM enc WHERE tbl IN {FACTS} ORDER BY 1, 2").fetchall():
    key = KEY[tbl]
    present = duckdb.sql("""
        SELECT count(*) FROM chunks WHERE arm = $arm AND "table" = $tbl AND path_in_schema = $key
    """, params={"arm": arm, "tbl": tbl, "key": key}).fetchone()[0]
    if not present:
        key_rows.append({"arm": arm, "table": tbl, "key": key, "row_groups": 0, "overlaps": None,
                         "verdict": "partitioned (key not in the file)"})
        continue
    r = duckdb.sql("""
        WITH r AS (
            SELECT TRY_CAST(any_value(stats_min_value) AS BIGINT) AS lo,
                   TRY_CAST(any_value(stats_max_value) AS BIGINT) AS hi
            FROM chunks
            WHERE arm = $arm AND "table" = $tbl AND path_in_schema = $key
              AND stats_min_value IS NOT NULL
            GROUP BY file_name, row_group_id
        ), o AS (SELECT lo, hi, lag(hi) OVER (ORDER BY lo, hi) AS prev_hi FROM r)
        SELECT count(*), count(*) FILTER (WHERE prev_hi IS NOT NULL AND prev_hi > lo) FROM o
    """, params={"arm": arm, "tbl": tbl, "key": key}).fetchone()
    verdict = ("no min/max stats on the key" if r[0] == 0 else
               "eliminable" if r[1] == 0 else f"{round(100.0 * r[1] / r[0])}% of neighbours overlap")
    key_rows.append({"arm": arm, "table": tbl, "key": key, "row_groups": r[0],
                     "overlaps": r[1], "verdict": verdict})
ordering = pd.DataFrame(key_rows)
display(ordering)

In [ ]:
# Write everything to the tpcds_bench lakehouse, Files section, so it can be read outside Fabric
# (`python fabric/pull_layout_stats.py --sf <sf>`). The raw per-arm footer reads are already on
# local disk as chunks_<arm>.parquet; the classified chunks and the summary tables join them, and
# duckrun uploads the folder in one call. overwrite=True: a re-run at the same sf replaces the set.
duckdb.sql(f"COPY enc TO '{OUT_DIR}/enc.parquet' (FORMAT PARQUET)")
for name, df in {"files": files_df, "summary_dict": summary_dict, "columns": columns_df,
                 "row_groups": row_groups, "ordering": ordering}.items():
    df.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)
print("staged:", sorted(os.listdir(OUT_DIR)))

bench = duckrun.connect(f"{ws_id}/{bench_id}", name="bench")
bench.copy(OUT_DIR, OUT, overwrite=True)
print(f"written to lakehouse {BENCH_LH}: Files/{OUT}")